In [0]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from mlflow.models.signature import infer_signature # <--- IMPORT THIS

gold_df = spark.read.table("shivam_catalog_f1_project.gold.pit_stop_features")
data_pd = gold_df.toPandas()
label = "pit_stop"
features = [
    "tire_age_laps",
    "lap_time_degradation",
    "lap_time_vs_race_avg",
    "lap_time_volatility",
    "lap_time"
]
X = data_pd[features]
y = data_pd[label]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
with mlflow.start_run() as run:
    print("Starting MLflow Run:", run.info.run_uuid)
    mlflow.set_tag("Model", "Logistic Regression with Balanced Weights")

    # --- Train the Model ---
    lr = LogisticRegression(random_state=42, class_weight='balanced')
    lr.fit(X_train, y_train)

    # --- Evaluate the Model ---
    predictions = lr.predict(X_test)
    
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    # --- Log Metrics ---
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    signature = infer_signature(X_train, predictions)
    mlflow.sklearn.log_model(lr, "pit-stop-predictor", signature=signature)
    print("\nModel and metrics logged to MLflow successfully (with signature).")

Starting MLflow Run: 41d0cac39c8e48a78f483982a70666e8
Accuracy: 0.9779
Precision: 0.4000
Recall: 1.0000
F1 Score: 0.5714


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(



Model and metrics logged to MLflow successfully (with signature).
🏃 View run clean-swan-901 at: https://adb-4462554101690101.1.azuredatabricks.net/ml/experiments/2021571639999054/runs/41d0cac39c8e48a78f483982a70666e8
🧪 View experiment at: https://adb-4462554101690101.1.azuredatabricks.net/ml/experiments/2021571639999054


In [0]:
import mlflow
catalog = "shivam_catalog_f1_project"
schema = "gold"
model_name = "pit_stop_predictor"
run_id = "41d0cac39c8e48a78f483982a70666e8" 
artifact_path = "pit-stop-predictor"
mlflow.set_registry_uri("databricks-uc")
source_uri = f"runs:/{run_id}/{artifact_path}"
destination_name = f"{catalog}.{schema}.{model_name}"
registered_model = mlflow.register_model(
    model_uri=source_uri,
    name=destination_name
)
print(f"Successfully registered model '{destination_name}' with version {registered_model.version}")

Registered model 'shivam_catalog_f1_project.gold.pit_stop_predictor' already exists. Creating a new version of this model...


Successfully registered model 'shivam_catalog_f1_project.gold.pit_stop_predictor' with version 1


Created version '1' of model 'shivam_catalog_f1_project.gold.pit_stop_predictor'.
